# NB01 — Do prompt ao agente

**Pergunta:** o que separa um prompt de um agente?  
**Caminho:** `v0` (agente sem ferramentas, com a política colada) → `v1` (ferramentas, decisão estruturada, estado, contexto enxuto, memória).

Cada seção mostra um problema real do RangoJá e a correção. As execuções vêm de **gravações do modelo real**; quando uma gravação foi escolhida entre várias, a célula diz a taxa.

> ▶ **Comece por aqui:** execute a célula de código abaixo **uma vez**, antes de qualquer seção. Ela importa tudo que os exemplos usam (`atender`, `verificar`, …). Depois disso, cada seção roda sozinha, na ordem que você quiser.

In [1]:
# Execute esta célula primeiro: ela importa o que todas as seções usam.
from rangoja import config, dados, falhas, ferramentas, controle, observabilidade, relogio
from rangoja.armazem import Armazem
from rangoja.casos import caso
from rangoja.execucao import atender, continuar, aprovar, recusar, expirar_pendencias, reiniciar_runner, estado_persistido
from rangoja.notebook import (preparar_ambiente, preparar_secao, mostrar, verificar, verificar_falha,
                              resumo, amostra, matriz)
from rangoja.trajetoria import chamou, ferramenta_antes, contar_chamadas
from rangoja.versoes import v0, v1, v2, v3, montar
preparar_ambiente()

modo,replay
provedor,openai
modelo,gpt-4o-mini
chave de API,sim
observabilidade,desligado (RANGOJA_PHOENIX=off)


## N1.1 — O prompt que funciona na demo

Caso `C01`: *"Meu pedido 4821 chegou frio e faltou o refri."* A `v0` recebe a política inteira na instrução e **nenhuma ferramenta**.

In [2]:
s = preparar_secao("N1.1")
mostrar(v0)
falha = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(falha)
amostra(s.gravacoes["falha"])

prompt,agente sem ferramentas; política na instrução


In [3]:
pedido = dados.pedido("4821")
print("itens do pedido 4821:", [(i.nome, float(i.preco), "entregue" if i.entregue else "NÃO entregue") for i in pedido.itens])
print("\nresposta da v0:\n", falha.resposta)
verificar_falha(falha, not falha.trajetoria, "a v0 não consultou nada: só tinha a política e a reclamação")
verificar_falha(falha, falha.decisao is None, "a resposta é só texto: não há decisão que o código possa usar")

itens do pedido 4821: [('Hambúrguer artesanal', 38.0, 'entregue'), ('Batata média', 18.0, 'entregue'), ('Refrigerante 2L', 8.0, 'NÃO entregue')]

resposta da v0:
 Lamento pela experiência que você teve. Seu pedido apresenta duas questões: 

1. Qualidade: O alimento chegou frio.
2. Item faltando: O refrigerante não foi entregue.

Para resolver, posso oferecer um voucher de R$ 15 pela qualidade do alimento e fazer o reembolso do valor do refrigerante. Você gostaria de seguir dessa forma?


O modelo só sabe o que está no contexto. Texto plausível não é fato. Como dar fatos ao modelo? **Ferramentas.**

## N1.2 — Ferramentas trazem fatos

A mesma reclamação, agora com ferramentas (`buscar_pedido`, `consultar_politica`, cupom, reembolso…). A docstring e os tipos de cada função são o que o modelo lê.

In [4]:
s = preparar_secao("N1.2")
mostrar(ferramentas._fabrica_buscar_pedido(enxuta=True))
correcao = atender(s.correcao, s.caso, gravacao=s.gravacoes["correcao"])
resumo(correcao)
amostra(s.gravacoes["correcao"])

agente,ferramenta,argumentos,efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 0, 'motivo': 'item_faltando'}",⛔ sem efeito
unico,oferecer_cupom,"{'pedido_id': '4821', 'valor': 15, 'motivo': 'qualidade'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 50, 'motivo': 'item_faltando'}",✅ com efeito


In [5]:
registrados = [c.args.get("valor") for c in correcao.trajetoria if c.ferramenta == "registrar_reembolso"]
print("consultou o pedido?", chamou(correcao.trajetoria, "buscar_pedido"))
print("reembolsos registrados:", registrados, "| valor real do refrigerante: R$ 8,00")
verificar_falha(correcao, correcao.erro is None and bool(correcao.trajetoria), "com ferramentas, o agente passou a agir no sistema")

consultou o pedido? False
reembolsos registrados: [0, 50] | valor real do refrigerante: R$ 8,00


**Achado das gravações:** ter ferramentas disponíveis **não garante** que o modelo as use para consultar os fatos. Nesta configuração (política colada na instrução, resposta em texto), o modelo agiu sem consultar o pedido em todas as execuções amostradas, e os valores que registrou não batem com o pedido. Veremos em N1.6 que o que muda isso é o que **entra no contexto**.

## N1.3 — O loop por baixo

Um agente é um loop: o modelo pede uma ferramenta, a ferramenta responde, o modelo decide o próximo passo ou responde. Os eventos do ADK mostram isso.

In [6]:
from rangoja.observabilidade import anatomia
s = preparar_secao("N1.3")
a = atender(s.correcao, s.caso, gravacao="N1.2/correcao")
print(anatomia(a))
passos = [ev for ev in a.eventos if ev.content and ev.content.parts]
verificar(len(passos) >= 2, "há pelo menos dois passos: o modelo age e o sistema responde")

atendimento C01 (v0+ferramentas) — status: concluido
├─ agente unico
│   ├─ pede ferramenta registrar_reembolso({'pedido_id': '4821', 'valor': 0, 'motivo': 'item_faltando'})
│   ├─ pede ferramenta oferecer_cupom({'pedido_id': '4821', 'valor': 15, 'motivo': 'qualidade'})
│   ├─ resultado de registrar_reembolso: {'erro': 'valor_invalido', 'total_do_pedido': '70.0'}
│   ├─ resultado de oferecer_cupom: {'cupom_id': 1, 'valor': 15, 'status': 'oferecido'}
│   ├─ pede ferramenta registrar_reembolso({'pedido_id': '4821', 'valor': 50, 'motivo': 'item_faltando'})
│   ├─ resultado de registrar_reembolso: {'reembolso_id': 4, 'valor': 50, 'status': 'registrado'}
│   └─ texto: '**Categoria:** item_faltando e qualidade  \n**Solução:** \n\n- O reembolso de R$ 50,00 pelo r'
└─ fonte das respostas do modelo: gravacao


## N1.4 — Prosa × estrutura

Caso `C02` (atraso). O agente decide em **prosa**: o código não tem como extrair categoria e valor. A correção é um **contrato**: a decisão é estruturada (`Decisao`) e a resposta ao cliente continua em texto.

In [7]:
s = preparar_secao("N1.4")
falha = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(falha); amostra(s.gravacoes["falha"])
verificar_falha(falha, falha.decisao is None, "em prosa, o código não recebe uma decisão utilizável")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '5310'},✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '5310', 'valor': 10, 'motivo': 'atraso de entrega'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '5310', 'valor': 7, 'motivo': 'atraso'}",✅ com efeito


In [8]:
from rangoja import tipos
mostrar(tipos.Decisao)
correcao = atender(s.correcao, s.caso, gravacao=s.gravacoes["correcao"])
resumo(correcao); amostra(s.gravacoes["correcao"])
verificar_falha(correcao, correcao.decisao is not None, "a decisão agora é um objeto tipado")
verificar_falha(correcao, correcao.decisao.solucao in __import__("rangoja.politica", fromlist=["x"]).solucoes_permitidas(correcao.decisao.categoria),
                "a solução decidida existe na política para a categoria")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '5310'},✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '5310', 'valor': 10, 'motivo': 'atraso na entrega'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '5310', 'valor': 7, 'motivo': 'atraso'}",✅ com efeito


Nota: com a API do ADK, a saída estruturada chega por uma ferramenta interna (`set_model_response`). Ela aparece nos eventos e é ignorada nas funções de trajetória.

## N1.5 — Estado da conversa

Caso `C11`: três mensagens (sem pedido → o 4822, faltou o refri → "e a sobremesa também"). Sem estado, cada mensagem abre uma sessão nova.

In [9]:
s = preparar_secao("N1.5")
falha = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(falha); amostra(s.gravacoes["falha"])
print("estado da última sessão (sem estado):", {k: v for k, v in falha.estado.items() if k in ("pedido_id", "compensacoes")})

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4822'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4822', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4822', 'valor': 10, 'motivo': 'atraso'}",✅ com efeito
unico,escalar_para_humano,"{'motivo': 'Recorrência de reembolsos', 'resumo': 'Cliente solicitou reembolso por item faltante (refrigerante 2L) e já possui 3 reembolsos nos últimos 30 dias.'}",✅ com efeito


estado da última sessão (sem estado): {}


In [10]:
correcao = atender(s.correcao, s.caso, gravacao=s.gravacoes["correcao"])
resumo(correcao); amostra(s.gravacoes["correcao"])
print("estado da sessão (com estado):", {k: v for k, v in correcao.estado.items() if k in ("pedido_id", "compensacoes")})
verificar_falha(correcao, correcao.estado.get("pedido_id") == "4822", "o estado guardou o pedido da conversa")
verificar_falha(falha, falha.estado.get("pedido_id") != "4822", "sem estado, a última sessão não sabe qual é o pedido")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4822'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4822', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4822', 'valor': 10, 'motivo': 'atraso na entrega'}",✅ com efeito


estado da sessão (com estado): {'pedido_id': '4822', 'compensacoes': 18.0}


## N1.6 — O que entra no contexto

Caso `C01`. Se `buscar_pedido` devolve **o pedido inteiro**, o endereço e os dados do entregador entram no contexto do modelo. A correção é devolver só o que a decisão precisa.

In [11]:
s = preparar_secao("N1.6")
falha = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(falha); amostra(s.gravacoes["falha"])
buscas = [c for c in falha.trajetoria if c.ferramenta == "buscar_pedido"]
print("campos devolvidos ao modelo (versão completa):", sorted(buscas[0].resultado.keys()))
verificar_falha(falha, "entregador" in buscas[0].resultado, "o contexto trouxe os dados do entregador (PII)")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 0, 'motivo': 'item_faltando'}",⛔ sem efeito
unico,oferecer_cupom,"{'pedido_id': '4821', 'valor': 15, 'motivo': 'entrega de item com qualidade abaixo do esperado'}",✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito


campos devolvidos ao modelo (versão completa): ['atraso_minutos', 'cliente_id', 'entregador', 'entregador_id', 'horarios', 'itens', 'pagamentos', 'pedido_id', 'restaurante', 'status', 'taxa_entrega', 'total']


In [12]:
correcao = atender(s.correcao, s.caso, gravacao=s.gravacoes["correcao"])
resumo(correcao); amostra(s.gravacoes["correcao"])
buscas = [c for c in correcao.trajetoria if c.ferramenta == "buscar_pedido"]
print("campos devolvidos ao modelo (versão enxuta):", sorted(buscas[0].resultado.keys()))
verificar_falha(correcao, "entregador" not in buscas[0].resultado, "com o contexto enxuto, nenhum dado do entregador chega ao modelo")

agente,ferramenta,argumentos,efeito
unico,buscar_pedido,{'pedido_id': '4821'},✅ com efeito
unico,consultar_politica,{'tema': 'item_faltando'},✅ com efeito
unico,registrar_reembolso,"{'pedido_id': '4821', 'valor': 8, 'motivo': 'item_faltando'}",✅ com efeito
unico,oferecer_cupom,"{'pedido_id': '4821', 'valor': 15, 'motivo': 'pedido chegou frio e item faltou'}",✅ com efeito


campos devolvidos ao modelo (versão enxuta): ['atraso_minutos', 'itens', 'pagamentos', 'pedido_id', 'restaurante', 'status', 'taxa_entrega', 'total']


## N1.7 — Lembrar do cliente

Caso `C13`: dois atendimentos do mesmo cliente. No primeiro ele diz que prefere a entrega na portaria. Sem memória, o segundo atendimento não sabe.

In [13]:
s = preparar_secao("N1.7")
falha = atender(s.falha, s.caso, gravacao=s.gravacoes["falha"])
resumo(falha); amostra(s.gravacoes["falha"])
verificar_falha(falha, "portaria" not in (falha.resposta or "").lower(), "sem memória, o segundo atendimento não menciona a portaria")

In [14]:
mostrar(ferramentas.anotar_observacao)
correcao = atender(s.correcao, s.caso, gravacao=s.gravacoes["correcao"])
resumo(correcao); amostra(s.gravacoes["correcao"])
print("observações guardadas para o cliente:", correcao.estado.get("user:observacoes"))
verificar_falha(correcao, "portaria" in (correcao.resposta or "").lower(), "com memória, o agente usa a preferência")

agente,ferramenta,argumentos,efeito
unico,consultar_politica,{'tema': 'entrega'},⛔ sem efeito
unico,anotar_observacao,{'texto': 'Cliente preferiu que a entrega fosse feita na portaria.'},✅ com efeito


observações guardadas para o cliente: ['Preferência de entrega na portaria do cliente.', 'Cliente preferiu que a entrega fosse feita na portaria.']


Atenção: **memória livre é uma porta de entrada**. No NB02 vemos o cliente usá-la para gravar um direito que não tem.

## N1.8 — Recapitulação

| Problema | Componente | Onde |
|---|---|---|
| O modelo inventa fatos | `ferramentas` | `rangoja/ferramentas.py` |
| Decisão em prosa | `schema` | `rangoja/tipos.py` (`Decisao`) |
| A conversa se perde | `estado` | `session.state` |
| Contexto cheio de dados que não servem | `contexto_enxuto` | `ferramentas._resumo_enxuto` |
| O agente não lembra do cliente | `memoria_livre` | `user:observacoes` |

**Ponte para o NB02:** o agente agora age e lembra. Nada ainda o impede de agir mal.